## PROJETO DE ANÁLISE DE ANÁLISE DE POSSIVEIS FRAUDES NO BANCO GLOBAL TRUCH

In [50]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, IsolationForest
import xgboost as xgb
from sklearn.svm import SVC
from xgboost import XGBClassifier
from catboost import CatBoostClassifier

from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, average_precision_score, roc_auc_score



In [36]:
#Plotando a base

base = pd.read_csv('Base_M43_Pratique_CREDIT_CARD_FRAUD.csv', delimiter=',')
base

,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,...,V21,V22,V23,V24,V25,V26,V27,V28,Amount,Class
0,0.0,-1.359807,-0.072781,2.536347,1.378155,-0.338321,0.462388,0.239599,0.098698,0.363787,...,-0.018307,0.277838,-0.110474,0.066928,0.128539,-0.189115,0.133558,-0.021053,149.62,0
1,0.0,1.191857,0.266151,0.166480,0.448154,0.060018,-0.082361,-0.078803,0.085102,-0.255425,...,-0.225775,-0.638672,0.101288,-0.339846,0.167170,0.125895,-0.008983,0.014724,2.69,0
2,1.0,-1.358354,-1.340163,1.773209,0.379780,-0.503198,1.800499,0.791461,0.247676,-1.514654,...,0.247998,0.771679,0.909412,-0.689281,-0.327642,-0.139097,-0.055353,-0.059752,378.66,0
3,1.0,-0.966272,-0.185226,1.792993,-0.863291,-0.010309,1.247203,0.237609,0.377436,-1.387024,...,-0.108300,0.005274,-0.190321,-1.175575,0.647376,-0.221929,0.062723,0.061458,123.50,0
4,2.0,-1.158233,0.877737,1.548718,0.403034,-0.407193,0.095921,0.592941,-0.270533,0.817739,...,-0.009431,0.798278,-0.137458,0.141267,-0.206010,0.502292,0.219422,0.215153,69.99,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
284802,172786.0,-11.881118,10.071785,-9.834783,-2.066656,-5.364473,-2.606837,-4.918215,7.305334,1.914428,...,0.213454,0.111864,1.014480,-0.509348,1.436807,0.250034,0.943651,0.823731,0.77,0
284803,172787.0,-0.732789,-0.055080,2.035030,-0.738589,0.868229,1.058415,0.024330,0.294869,0.584800,...,0.214205,0.924384,0.012463,-1.016226,-0.606624,-0.395255,0.068472,-0.053527,24.79,0
284804,172788.0,1.919565,-0.301254,-3.249640,-0.557828,2.630515,3.031260,-0.296827,0.708417,0.432454,...,0.232045,0.578229,-0.037501,0.640134,0.265745,-0.087371,0.004455,-0.026561,67.88,0
284805,172788.0,-0.240440,0.530483,0.702510,0.689799,-0.377961,0.623708,-0.686180,0.679145,0.392087,...,0.265245,0.800049,-0.163298,0.123205,-0.569159,0.546668,0.108821,0.104533,10.00,0


**TIME** :  Tempo decorrido na transação.

**V1 até V28** :  Variáveis numéricas tranformadadas por PCA. 

**AMOUNT** :  Valor da transação.

**CLASS** :  variável-alvo, sendo 0 = Legítima e 1 = Fraude.

In [ ]:
# Podemos facilmente vizualizar que os dados estão com PCA (Pricipal Component Analysis), ou seja, tranformado em números, por um lado, não temos dados específicos, como idade do cliente, localização ou tipo de compra.
# Por outro lado, os modelos de Machine Learning lidam melhor com dados numéricos, podendo até mesmo facilitar a distinção entre transações legítimas e fraudes.
# A variável TIME, Amount e Class não utilizam PCA.

## ANÁLISE EXPLORATÓRIA 

In [34]:
print("Teste do Python funcionando!")

Teste do Python funcionando!


In [37]:
base.isnull().sum()

Time      0
V1        0
V2        0
V3        0
V4        0
V5        0
V6        0
V7        0
V8        0
V9        0
V10       0
V11       0
V12       0
V13       0
V14       0
V15       0
V16       0
V17       0
V18       0
V19       0
V20       0
V21       0
V22       0
V23       0
V24       0
V25       0
V26       0
V27       0
V28       0
Amount    0
Class     0
dtype: int64

In [38]:
base.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 284807 entries, 0 to 284806
Data columns (total 31 columns):
 #   Column  Non-Null Count   Dtype  
---  ------  --------------   -----  
 0   Time    284807 non-null  float64
 1   V1      284807 non-null  float64
 2   V2      284807 non-null  float64
 3   V3      284807 non-null  float64
 4   V4      284807 non-null  float64
 5   V5      284807 non-null  float64
 6   V6      284807 non-null  float64
 7   V7      284807 non-null  float64
 8   V8      284807 non-null  float64
 9   V9      284807 non-null  float64
 10  V10     284807 non-null  float64
 11  V11     284807 non-null  float64
 12  V12     284807 non-null  float64
 13  V13     284807 non-null  float64
 14  V14     284807 non-null  float64
 15  V15     284807 non-null  float64
 16  V16     284807 non-null  float64
 17  V17     284807 non-null  float64
 18  V18     284807 non-null  float64
 19  V19     284807 non-null  float64
 20  V20     284807 non-null  float64
 21  V21     28

In [39]:
# Os dados passaram por um pré-processamento, ao analisar dados faltantes, todas as colunas estão preenchidas.

In [40]:
base.dtypes

Time      float64
V1        float64
V2        float64
V3        float64
V4        float64
V5        float64
V6        float64
V7        float64
V8        float64
V9        float64
V10       float64
V11       float64
V12       float64
V13       float64
V14       float64
V15       float64
V16       float64
V17       float64
V18       float64
V19       float64
V20       float64
V21       float64
V22       float64
V23       float64
V24       float64
V25       float64
V26       float64
V27       float64
V28       float64
Amount    float64
Class       int64
dtype: object

In [41]:
# Os dados estão todos no mesmo tipo de dados

In [42]:
base.duplicated().sum()

np.int64(1081)

In [43]:
# Para melhorar a base de dados, utilizei a linha de código acima para ver se há dados duplicados e atuar para remover essas duplicatas.

In [44]:
base.drop_duplicates(inplace=True)

In [12]:
# Também não há interesse em realizar um comparativo entre a variável alvo com as demais variáveis, pois como os dados então em formato PCA, a análise exploratória fica quase impossivel de ser realizada.
# Sendo assim, o principal objetivo é, mediante a desproporcionalidade das fraudes quanto as transações legítimas, criar um modelo capaz de prever com acertvidade essas fraudes.

## Separação treino e teste e aplicação em modelos de Machine Learning

In [13]:
# Separação em treino e teste para aplicarmos em diferentes modelos para alcançar o melhor resultado possivel, aplicando hiperparametros em cada modelo.

In [45]:
X = base.drop('Class', axis=1)
Y = base['Class']

In [46]:
# Separação de base treino e teste 
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, random_state=42, stratify=Y)
# stratify esta sendo usado pelo fato de haver poucas fraudes, esse metodo ajuda a manter a proporção entre classes nos conjuntos de treino e teste.

print("\nTreino:", X_train.shape)
print("Teste:", X_test.shape)
print("Fraudes no treino:", int(Y_train.sum()))
print("Fraudes no teste:", int(Y_test.sum()))


Treino: (226980, 30)
Teste: (56746, 30)
Fraudes no treino: 378
Fraudes no teste: 95


In [16]:
# Podemos notar o quão discrepante é o volume de fraudes em comparação dos dados como um todo. 

## Regressão Logistica

In [17]:
# Regressão Logística levando em consiferação os principais hiperparametros
modelo_lr = Pipeline([
    ("scaler", StandardScaler()),
    ("modelo", LogisticRegression(
        class_weight="balanced",
        max_iter=2000,
        random_state=42
    ))
])

In [18]:
# Treinamento do modelo
modelo_lr.fit(X_train, Y_train)

# Prever as classes e as probabilidades
y_pred_lr = modelo_lr.predict(X_test)
y_prob_lr = modelo_lr.predict_proba(X_test)[:, 1]

In [19]:
# Matriz de confusão
print("Matriz de confusão:")
print(confusion_matrix(Y_test, y_pred_lr))

# Métricas de classificação
print("\nRelatório de classificação:")
print(classification_report(
    Y_test,
    y_pred_lr,
    digits=4,
    zero_division=0
))

# Métricas gerais
ap_lr = average_precision_score(Y_test, y_prob_lr)
auc_lr = roc_auc_score(Y_test, y_prob_lr)

print(f"Average Precision: {ap_lr:.4f}")
print(f"ROC-AUC: {auc_lr:.4f}")


Matriz de confusão:
[[55262  1389]
 [   12    83]]

Relatório de classificação:
              precision    recall  f1-score   support

           0     0.9998    0.9755    0.9875     56651
           1     0.0564    0.8737    0.1059        95

    accuracy                         0.9753     56746
   macro avg     0.5281    0.9246    0.5467     56746
weighted avg     0.9982    0.9753    0.9860     56746

Average Precision: 0.6719
ROC-AUC: 0.9657


In [20]:
# Pdemos notar que a falta de um volume considerável de fraudes impacta diretamente no modelo.
# A precisão do modelo foi de menos de 10%
# Já o Recall teve uma acertividade de mais de 90%, ou seja, das 98 fraudes detectadas, sua acertividade para identificar transações fraudulentas foram significativas.
# A acurácia ficou acima do esperado, atingindo 97%, um bom desempenho.

## Random Forest

In [21]:
# Random Forest levando em consideração os principais hiperparametros
modelo_rf = RandomForestClassifier(
        n_estimators=200,
        class_weight="balanced_subsample",
        random_state=42,
        n_jobs=-1
    )

# Treinamento do modelo
modelo_rf.fit(X_train, Y_train)

,n_estimators,200
,criterion,'gini'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [22]:
# Prever as classes  e obter as probabilidades de fraude
y_pred_rf = modelo_rf.predict(X_test)
y_prob_rf = modelo_rf.predict_proba(X_test)[:, 1]

In [23]:
# Matriz de confusão
print("Matriz de confusão:")
print(confusion_matrix(Y_test, y_pred_lr))

# Métricas de classificação
print("\nRelatório de classificação:")
print(classification_report(
    Y_test, y_pred_rf,
    digits=4,
    zero_division=0
))

print(f"Average Precision:",
      round(average_precision_score(Y_test, y_prob_rf), 4))
print(f"ROC-AUC:",
      round(roc_auc_score(Y_test, y_prob_rf), 4))

Matriz de confusão:
[[55262  1389]
 [   12    83]]

Relatório de classificação:
              precision    recall  f1-score   support

           0     0.9995    1.0000    0.9997     56651
           1     0.9710    0.7053    0.8171        95

    accuracy                         0.9995     56746
   macro avg     0.9853    0.8526    0.9084     56746
weighted avg     0.9995    0.9995    0.9994     56746

Average Precision: 0.8061
ROC-AUC: 0.9444


In [24]:
# O Random Forest teve um exelente desempenho, mostrando resultados melhores que a Regressão Logística.
# Precisão está acima de 96%, oque mostra que o desempenho para classificar fraude foi muito positivo.
# Recall teve uma desempenho levemente positivo, estando abaixo dos 75%, e o F1-Score que mede a harmonia entre precision e recall, somando mais de 83%.
# Por fim, a accuracy teve um exelente resultado, batendo quase 100%, mostrando que os hiperparametros fizeram diferença.

## XGBoost

In [25]:
# XGboost levando em consideração os principais hiperparametros
# Calcular o peso da classe minoritária.
peso_fraude = (Y_train == 0).sum() / (Y_train == 1).sum()

# Construção do modelo
modelo_xgb = XGBClassifier(
        n_estimators=200,
        max_depth=5,
        learning_rate=0.05,
        scale_pos_weight=peso_fraude,
        eval_metric="aucpr",
        random_state=42,
        n_jobs=-1
    )

# treinamento com X_train e Y_train
modelo_xgb.fit(X_train, Y_train)

,objective,'binary:logistic'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,None
,device,None
,early_stopping_rounds,None
,enable_categorical,True
,eval_metric,'aucpr'


In [26]:
# Prever as classes  e obter as probabilidades de fraude
y_pred_xgb = modelo_xgb.predict(X_test)
y_prob_xgb = modelo_xgb.predict_proba(X_test)[:, 1]

In [27]:
# Matriz de confusão
print("Matriz de confusão:")
print(confusion_matrix(Y_test, y_pred_lr))

# Métricas de classificação
print("\nRelatório de classificação:")
print(classification_report(
    Y_test, y_pred_xgb,
    digits=4,
    zero_division=0
))

print(f"Average Precision:",
      round(average_precision_score(Y_test, y_prob_rf), 4))
print(f"ROC-AUC:",
      round(roc_auc_score(Y_test, y_prob_rf), 4))

Matriz de confusão:
[[55262  1389]
 [   12    83]]

Relatório de classificação:
              precision    recall  f1-score   support

           0     0.9997    0.9992    0.9994     56651
           1     0.6179    0.8000    0.6972        95

    accuracy                         0.9988     56746
   macro avg     0.8088    0.8996    0.8483     56746
weighted avg     0.9990    0.9988    0.9989     56746

Average Precision: 0.8061
ROC-AUC: 0.9444


In [28]:
# O modelo teve ótimos números para os transações legitimas, onde em todas as métricas tivemos resultados acima dos 98%.
# Agora para prever fraudes (1), o recall teve 85% das fraudes identificadas, entretanto a precisão foi de 52%, muito abaixo se comparado ao modelo anterior.
# Accuracy teve resultados satisfatório, tendo 99% de pontuação. 

## CatBoost

In [29]:
# CatBoostClassifier levando em consideração os principais hiperparamentros
modelo_cat = CatBoostClassifier(
        iterations=200,
        depth=6,
        learning_rate=0.05,
        auto_class_weights="Balanced",
        verbose=0,
        random_seed=42,
        allow_writing_files=False
    )
# Treinamento do modelo 
modelo_cat.fit(X_train, Y_train)

CatBoostClassifier(allow_writing_files=False, auto_class_weights='Balanced', depth=6, iterations=200, learning_rate=0.05, random_seed=42, verbose=0)

In [30]:
y_pred_cat = modelo_cat.predict(X_test).ravel().astype(int)
y_prob_cat = modelo_cat.predict_proba(X_test)[:, 1]

In [31]:
# Matriz de confusão
print("Matriz de confusão:")
print(confusion_matrix(Y_test, y_pred_lr))

# Métricas de classificação
print("\nRelatório de classificação:")
print(classification_report(
    Y_test, y_pred_cat,
    digits=4,
    zero_division=0
))

print(f"Average Precision:",
      round(average_precision_score(Y_test, y_prob_rf), 4))
print(f"ROC-AUC:",
      round(roc_auc_score(Y_test, y_prob_rf), 4))

Matriz de confusão:
[[55262  1389]
 [   12    83]]

Relatório de classificação:
              precision    recall  f1-score   support

           0     0.9997    0.9992    0.9994     56651
           1     0.6260    0.8105    0.7064        95

    accuracy                         0.9989     56746
   macro avg     0.8128    0.9049    0.8529     56746
weighted avg     0.9991    0.9989    0.9989     56746

Average Precision: 0.8061
ROC-AUC: 0.9444


In [32]:
# O modelo se comportou bem para transações leg[itimas, para identificar fraudes tivemos resultados abaixo dos 90%.
# Precision teve 58%, recall tivemos 87%, um saldo positivo para identificar quantas transações realmente foram fraudads.
# Accuracy se manteve alta neste modelo, berando os 100%.

## Isolation Forest

In [ ]:
o Isolation Forest é utilizado para detectar anomalias ou compartamentos diferentes da maioria, oque faz sentido na detecção de possíves fraudes em transações financeiras.

In [47]:
# Selecionar transações legítimas para treinamento específico desse modelo
X_train_legitimas = X_train.loc[Y_train== 0]

# Criação do modelo
modelo_if = IsolationForest(
    n_estimators=200,
    contamination="auto",
    random_state=42,
    n_jobs=-1
)

# Treinamento
modelo_if.fit(X_train_legitimas)

,n_estimators,200
,max_samples,'auto'
,contamination,'auto'
,max_features,1.0
,bootstrap,False
,n_jobs,-1
,random_state,42
,verbose,0
,warm_start,False


In [48]:
# Isolation forest retorna
# -1 = anomalia
#  1 = comportamento normal
pred_if_original = modelo_if.predict(X_test)

# conversão para:
# 1 = possivel fraude
# 0 = transação normal
y_pred_if = (pred_if_original == -1).astype(int)

# Pontuação contínua de anomalia
y_score_if = -modelo_if.decision_function(X_test)


In [49]:
print("Matriz de confusão:")
print(confusion_matrix(Y_test, y_pred_if))

print("\nRelatório de classificação:")
print(classification_report(
    Y_test, y_pred_if,
    digits=4,
    zero_division=0
))

print("Average Precision:",
      round(average_precision_score(Y_test, y_score_if), 4))

print("ROC-AUC:",
      round(roc_auc_score(Y_test, y_score_if), 4))

Matriz de confusão:
[[54409  2242]
 [   20    75]]

Relatório de classificação:
              precision    recall  f1-score   support

           0     0.9996    0.9604    0.9796     56651
           1     0.0324    0.7895    0.0622        95

    accuracy                         0.9601     56746
   macro avg     0.5160    0.8749    0.5209     56746
weighted avg     0.9980    0.9601    0.9781     56746

Average Precision: 0.0665
ROC-AUC: 0.9318


In [ ]:
# O isolation forest não teve uma boa precision, estando abaixo de 10%.
# O modelo se comportou bem, apresentando 85% de recall e menos de 10% de f1-score.
# Accuracy de 95%, muito puxado pela boa acertividade das transações lçegiítimas, estando acima de 95%.

## Análise dos 5 modelos e aprimoramento do modelo escolhido

In [ ]:
# O modelo que melhor trouxe resultados foi o Random forest, o modelo foi capaz de alcançar uma precisão de 97%, gerando alertas mais confiáveis de fraudes e alcaçou um recall de 70%, quando o modelo acusou fraude, 70% era verdadeiros posiivos.
# O modelo sofre com a falta de um maior volume de fraudes para identificar padrões, por isso optei em priorizar o modelo cuja precisão fosse alta.       

In [ ]:
# Utilizamos hiperparametros para aprimorar o modelo RANDOM FOREST, agora vou utlizar validação cruzada para deixar o modelo preparado para se adaptar melhor para generalização dos daos.

## Validação Cruzada

In [59]:
# Configuração da validação cruzada
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

In [60]:
# Criação do modelo Random Forest com Cross Validation
modelo_rfcv = RandomForestClassifier(
    n_estimators=200,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

# Definir as métricas
metricas = {
    "precision": "precision",
    "recall": "recall",
    "f1": "f1",
    "average_precision": "average_precision"
}

# Executar Cross-Validation
resultados = cross_validate(
    modelo_rfcv,
    X_train,
    Y_train,
    cv=cv,
    scoring=metricas,
    n_jobs=1,
    return_train_score=False
)

# 5. Exibir resultados de cada métrica
for metrica in metricas:
    valores = resultados[f"test_{metrica}"]

    print(f"\n{metrica.upper()}")
    print(f"Média: {valores.mean():.4f}")
    print(f"Desvio-padrão: {valores.std():.4f}")
    print(f"Resultados por fold: {np.round(valores, 4)}")



PRECISION
Média: 0.9421
Desvio-padrão: 0.0219
Resultados por fold: [0.9643 0.9672 0.9412 0.9091 0.9286]

RECALL
Média: 0.7566
Desvio-padrão: 0.0757
Resultados por fold: [0.72   0.7867 0.6316 0.7895 0.8553]

F1
Média: 0.8367
Desvio-padrão: 0.0460
Resultados por fold: [0.8244 0.8676 0.7559 0.8451 0.8904]

AVERAGE_PRECISION
Média: 0.8388
Desvio-padrão: 0.0315
Resultados por fold: [0.8424 0.8629 0.7783 0.846  0.8643]


In [ ]:
# O RANDOM FOREST teve um melhor desempenho após aplicar validação cruzada, onde em diferentes partições de dados, a precisão se manteve acima dos 90%
# o Recall teve media alta de 83%, e o equilibrio entre RECALL e PRECISION (F1-SCORE), se mante acima dos 83%, sendo o melhor modelo para prever fraudes.